# freqk_gr — Explore Allele Frequency Results

Reads in all completed `allele_frequencies.k31.tsv` files from `results/`,  
computes ALT allele frequency per variant per sample, and does basic EDA.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path

# ── paths ──────────────────────────────────────────────────────────────────
RESULTS = Path("/home/tbellagio/scratch/freqk_gr/results")
K = 31

# NOTE: we use counts_by_allele (ref_kmers|alt_kmers) rather than
# allele_frequencies, because freqk call outputs binary 0/1 majority-vote
# calls, not continuous frequencies. AF = alt / (ref + alt).

## 1. Discover completed samples

In [ ]:
af_files = sorted(RESULTS.glob(f"*/*/k{K}/*.counts_by_allele.k{K}.tsv"))
print(f"Completed samples: {len(af_files)}")
for f in af_files:
    print(" ", f.parent.parent.name)

## 2. Load all results into a single DataFrame

In [ ]:
def parse_sample_id(sample_id):
    """Extract site, plot, date, year from MLFH{site}{plot}{date}."""
    site  = sample_id[4:6]
    plot  = sample_id[6:8]
    date  = sample_id[8:16]
    year  = sample_id[8:12]
    return site, plot, date, year

def load_af(path, k=K):
    """Load counts_by_allele file, compute AF = alt / (ref + alt)."""
    sample_id = path.parent.parent.name
    site, plot, date, year = parse_sample_id(sample_id)

    df = pd.read_csv(path, sep="|", header=None, names=["ref", "alt"], dtype=float)
    df[["ref", "alt"]] = df[["ref", "alt"]].fillna(0).astype(int)

    df["sample_id"]   = sample_id
    df["site"]        = site
    df["plot"]        = plot
    df["date"]        = pd.to_datetime(date, format="%Y%m%d")
    df["year"]        = year
    df["variant_idx"] = df.index  # 0-based row = variant index

    total = df["ref"] + df["alt"]
    df["af"] = np.where(total > 0, df["alt"] / total, np.nan)
    return df

dfs = [load_af(f) for f in af_files]
df = pd.concat(dfs, ignore_index=True)
print(df.shape)
df.head()

## 3. Basic QC

In [ ]:
df["coverage"] = df["ref"] + df["alt"]
covered = df[df["af"].notna() & (df["coverage"] > 0)]

# variants with zero coverage (ref=0 and alt=0) → uncovered
n_uncovered = df.groupby("sample_id").apply(lambda x: (x["coverage"] == 0).sum())
n_total     = df.groupby("sample_id").size()

qc = pd.DataFrame({"total": n_total, "uncovered": n_uncovered})
qc["pct_uncovered"] = 100 * qc["uncovered"] / qc["total"]
qc = qc.sort_values("pct_uncovered")
print(qc.to_string())

In [ ]:
# AF distribution per sample — only covered variants
fig, ax = plt.subplots(figsize=(10, 4))
for sid, grp in covered.groupby("sample_id"):
    grp["af"].plot.hist(bins=50, alpha=0.4, label=sid, ax=ax, density=True)
ax.set_xlabel("ALT allele frequency")
ax.set_ylabel("Density")
ax.set_title("AF distribution per sample")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

## 4. Variant coverage (ref + alt k-mer counts)

In [ ]:
df["coverage"] = df["ref"] + df["alt"]
covered = df[df["af"].notna() & (df["coverage"] > 0)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# coverage distribution
covered["coverage"].clip(upper=200).plot.hist(
    bins=50, ax=axes[0], color="steelblue", edgecolor="white"
)
axes[0].set_xlabel("Total k-mer coverage (ref + alt, clipped at 200)")
axes[0].set_ylabel("Variants")
axes[0].set_title("Coverage distribution (all samples)")

# median coverage per sample
med_cov = covered.groupby("sample_id")["coverage"].median().sort_values()
med_cov.plot.barh(ax=axes[1], color="steelblue")
axes[1].set_xlabel("Median k-mer coverage")
axes[1].set_title("Median coverage per sample")

plt.tight_layout()
plt.show()

## 5. AF over time (site 04)

In [ ]:
# mean ALT AF per time point across plots — site 04 only
site04 = covered[covered["site"] == "04"]

time_af = (
    site04
    .groupby(["variant_idx", "date"])["af"]
    .mean()
    .reset_index()
)

# summarise across variants: mean ± std
summary = time_af.groupby("date")["af"].agg(["mean", "std"]).reset_index()

fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(summary["date"], summary["mean"], yerr=summary["std"],
            fmt="o-", capsize=4, color="steelblue")
ax.set_xlabel("Date")
ax.set_ylabel("Mean ALT allele frequency")
ax.set_title("Site 04 — mean AF across variants over time")
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%Y-%m-%d"))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 6. Per-plot AF comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(
    data=site04,
    x="date", y="af", hue="plot",
    ax=ax, flierprops=dict(marker=".", alpha=0.2, markersize=1)
)
ax.set_xlabel("Date")
ax.set_ylabel("ALT allele frequency")
ax.set_title("Site 04 — AF per plot per time point")
ax.legend(title="Plot", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()